# Advanced EDA: Cumulative Analysis & Moving Aggregations

## Executive Summary & Analytical Domain
This notebook computes **Cumulative Running Totals** and **Moving Averages** across historical sales data from `gold.fact_sales`. Cumulative analytics are crucial for:
- Tracking cumulative milestone achievements against enterprise revenue targets.
- Evaluating long-term revenue acceleration versus deceleration.
- Smoothing out short-term pricing volatility through rolling average selling prices.

## Key Performance Indicators (KPIs) & Methodology
- **Monthly Revenue (`total_sales`)**: Discrete sales recorded in each calendar month.
- **Running Total Sales (`running_total_sales`)**: Cumulative cumulative sum from the inception of trading up to the current period (`SUM(...) OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`).
- **Moving Average Selling Price (`moving_avg_price`)**: Progressive mean price per unit over time (`AVG(...) OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`).

## Bug Fixes & Dialect Modernization
- **Modular Query Architecture**: Implements clean ANSI/Spark Common Table Expressions (CTEs) for cumulative running metrics.nal PostgreSQL script contained a duplicate `SELECT \n SELECT` statement; this has been corrected to a clean ANSI/Spark Common Table Expression (CTE).
- **Derived Table Aliasing**: Subqueries in FROM clauses now follow strict Spark SQL aliasing standards.
- **Deterministic Window Frames**: Explicit `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` frame definitions guarantee deterministic cumulative aggregation.

## Data Contracts & Layer Dependencies
- **Target Layer**: `gold`
- **Primary Source Table**: `gold.fact_sales` (`order_date`, `sales_amount`, `price`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. Monthly Cumulative Revenue & Moving Average Price

### Business Context & Objective
Progressively aggregates monthly sales revenue and moving average item prices to observe growth velocity and long-term price realization trends.

In [ ]:
# COMMAND ----------
# 1. Monthly Cumulative Analysis with Window Functions
df_monthly_cumulative = spark.sql("""
WITH monthly_sales AS (
    SELECT
        date_trunc('month', order_date) AS order_date,
        sum(sales_amount) AS total_sales,
        avg(price) AS avg_price
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY date_trunc('month', order_date)
)
SELECT
    order_date,
    total_sales,
    sum(total_sales) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_sales,
    avg(avg_price) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS moving_avg_price
FROM monthly_sales
ORDER BY order_date
""")
display(df_monthly_cumulative)

### Strategic Insights & Interpretation (Monthly Cumulative)
- **Cumulative Milestone Tracking**: Demonstrates the exact timeline at which cumulative revenue surpassed major milestones ($1M, $5M, $10M).
- **Moving Average Price Stability**: Identifies whether product pricing has remained resilient over time or experienced downward price erosion due to discounts and lower-tier inventory mixes.

## 2. Annual Cumulative Trajectory

### Business Context & Objective
Aggregates performance on an annual cadence to provide executive leadership with a multi-year macroeconomic view of cumulative capital generation.

In [ ]:
# COMMAND ----------
# 2. Annual Cumulative Analysis
df_yearly_cumulative = spark.sql("""
WITH yearly_sales AS (
    SELECT
        date_trunc('year', order_date) AS order_date,
        sum(sales_amount) AS total_sales,
        avg(price) AS avg_price
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY date_trunc('year', order_date)
)
SELECT
    order_date,
    total_sales,
    sum(total_sales) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_sales,
    avg(avg_price) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS moving_avg_price
FROM yearly_sales
ORDER BY order_date
""")
display(df_yearly_cumulative)

### Business Recommendations & Summary
1. **Growth Pacing**: Use cumulative slope analysis to determine whether enterprise expansion is linear, exponential, or plateauing.
2. **Pricing Policy**: Monitor moving average prices against inflationary pressure to protect gross margin targets.
3. **Financial Projections**: Leverage historical cumulative run rates to establish realistic multi-year revenue guidance.